In [0]:
CATALOGOS = ["electrocasa_dev", "electrocasa_prod"]

GRUPO_ING = "electrocasa_ingenieria"
GRUPO_ANALISTAS = "electrocasa_analistas"
GRUPO_AUDITORIA = "electrocasa_auditoria"

print("Grupos esperados:")
print("-", GRUPO_ING)
print("-", GRUPO_ANALISTAS)
print("-", GRUPO_AUDITORIA)


In [0]:
for catalogo in CATALOGOS:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalogo}")

    for schema in ["landing", "bronze", "silver", "gold", "audit"]:
        spark.sql(
            f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema}"
        )

    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS {catalogo}.bronze.landing"
    )

    print(f"Aprovisionado: {catalogo}")


In [0]:
for catalogo in CATALOGOS:
    spark.sql(
        f"GRANT USE CATALOG ON CATALOG {catalogo} TO `{GRUPO_ING}`"
    )

    for schema in ["landing", "bronze", "silver", "gold", "audit"]:
        spark.sql(
            f"GRANT USE SCHEMA ON SCHEMA {catalogo}.{schema} "
            f"TO `{GRUPO_ING}`"
        )
        spark.sql(
            f"GRANT SELECT, MODIFY, CREATE TABLE "
            f"ON SCHEMA {catalogo}.{schema} TO `{GRUPO_ING}`"
        )

    spark.sql(
        f"GRANT READ VOLUME, WRITE VOLUME "
        f"ON VOLUME {catalogo}.bronze.landing TO `{GRUPO_ING}`"
    )

    spark.sql(
        f"GRANT USE CATALOG ON CATALOG {catalogo} "
        f"TO `{GRUPO_ANALISTAS}`"
    )
    spark.sql(
        f"GRANT USE SCHEMA ON SCHEMA {catalogo}.gold "
        f"TO `{GRUPO_ANALISTAS}`"
    )
    spark.sql(
        f"GRANT SELECT ON SCHEMA {catalogo}.gold "
        f"TO `{GRUPO_ANALISTAS}`"
    )

    spark.sql(
        f"GRANT USE CATALOG, BROWSE ON CATALOG {catalogo} "
        f"TO `{GRUPO_AUDITORIA}`"
    )

    for schema in ["gold", "audit"]:
        spark.sql(
            f"GRANT USE SCHEMA ON SCHEMA {catalogo}.{schema} "
            f"TO `{GRUPO_AUDITORIA}`"
        )
        spark.sql(
            f"GRANT SELECT ON SCHEMA {catalogo}.{schema} "
            f"TO `{GRUPO_AUDITORIA}`"
        )

    print(f"Permisos aplicados: {catalogo}")


In [0]:
for catalogo in CATALOGOS:
    ruta = f"/Volumes/{catalogo}/bronze/landing"
    print(ruta)
    display(dbutils.fs.ls(ruta))
